In [ ]:
from dbm import error

import src.config as config
import pickle
with open("../" + config.datasets_path + "/{}.dataset".format("no_bias"), "rb") as f:
    X, Y  = pickle.load(f)
X

In [ ]:
with open("../" + config.datasets_path + "/{}.preprocessed".format("historical_biased_y"), "rb") as f:
    X, Y  = pickle.load(f)
X

In [ ]:
with open(f"../../rsc/preprocessors/no_bias.preprocessor", "rb") as f:
    preprocessor  = pickle.load(f)

In [ ]:
preprocessor

# Generating a dataset with no bias


In [ ]:
#!pip3 install biasondemand
import biasondemand
import numpy as np
import pandas as pd
from numpy.random.mtrand import seed

biasondemand.generate_dataset(path='my_unbiased_dataset', dim=1000, l_m=0.0)

**Comment**

It generates a folder with:
- `X_train.csv`: containing the training data (67%);
- `y_train.csv`: containing the training labels;
- `X_test.csv`: containing the test data (33%);
- `y_test.csv`: containing the test labels;

and, in addition, with:
- `X_ind_test`: Individual test set (without A);
- `X_supp_test`: Suppression test set (without variable correlated with A);
- `y_test_real` : Target test - no measurement bias;

                    Dataset biased
                         │
              ┌──────────┼──────────┐
              ↓          ↓          ↓
           Full       Individual  Suppression
              │          │          │
              │          │          └─ elimina feature
              │          │             troppo correlate
              │          │             con A
              │          │
              │          └─ elimina A
              │
              └─ mantiene A

In [ ]:
X_train = pd.read_csv('../spikes/datasets/my_unbiased_dataset/X_train.csv', index_col=0)
y_train = pd.read_csv('../spikes/datasets/my_unbiased_dataset/y_train.csv', index_col=0)
pd.concat([X_train, y_train], axis=1)

- $Y$: target variable;
- $R$: resources of individuals relevant to the problem (they directly impact $Y$), note that if `l_m != 0` $P$ substitutes $R$;
- $A$: sensitive attribute;
- $P_R$: proxy variables of $R$;
- $Q$: additional variables that may or may not impact $Y$ and that might be (or not) impacted by $R$ or $A$.

In [ ]:
X_supp_train = pd.read_csv('../spikes/datasets/my_unbiased_dataset/X_supp_train.csv', index_col=0)
X_supp_train

# Generating a Biased dataset
You can introduce different types of bias into the dataset by specifying command line arguments. For example, to generate a dataset with measurement bias on the label Y (magnitude: 1.5) and historical bias on the feature R (magnitude: 2), use the following command:


In [ ]:
biasondemand.generate_dataset(path='my_biased_dataset', dim=1000, l_m_y=1.5, l_h_r=2)
X_train = pd.read_csv('../spikes/datasets/my_biased_dataset/X_train.csv', index_col=0)
y_train = pd.read_csv('../spikes/datasets/my_biased_dataset/y_train.csv', index_col=0)
pd.concat([X_train, y_train], axis=1)

Notice that the biases are introduced w.r.t. idividuals in the group A=1. For most types of bias, larger values mean more bias. The only exceptions are undersampling and representation bias (which can be seen as a specific type of undersampling conditional on the feature R) where smaller values correspond to more (conditional) undersampling, i.e., more bias.


**Recap of parameters**

Properties of the dataset:
- `dim`: Dimension of the dataset;
- `sy`: Standard deviation of the noise of Y. I guess this refers to $\epsilon$ in the paper;
- `l_q`: Lambda coefficient for importance of Q for Y;
- `l_r_q`: Lambda coefficient that quantifies the influence from R to Q;
- `thr_supp`: Threshold correlation for discarding features too much correlated with s;

Types of biases:
- `l_y`: Lambda coefficient for **historical bias** on the target y
- `l_h_r`: Lambda coefficient for **historical bias** on R
- `l_h_q`: Lambda coefficient for **historical bias** on Q
- `l_m_y`: Lambda coefficient for **measurement bias** on the target y
- `l_m`: Lambda coefficient for **measurement bias** on the feature R. If `l_m!=0` P substitutes R.
- `p_u`: Percentage of **undersampling** instance with A=1
- `l_r`: Boolean for inducing **representation bias**, that is undersampling conditioning on a variable, e.g. R
- `l_o`: Boolean variable for excluding an important variable (**ommited variable bias**), e.g. R (or its proxy)
- `l_y_b`: Lambda coefficient for **interaction proxy bias**, i.e., **historical bias** on the label y with lower values of y for individuals in group A=1 with high values for the feature R

---

- l_h_r   → bias storico su R
- l_h_q   → bias storico su Q
- l_y     → bias storico su Y
- l_m_y   → measurement bias su Y
- l_r     → representation bias
- l_o     → omission bias
- l_y_b   → interaction/proxy bias


| Parametro  | Min/max tecnici dal codice | Cosa controlla                          | Valori sensati da provare |
| ---------- | -------------------------: | --------------------------------------- | ------------------------: |
| `dim`      |                 `1+` circa | numero di istanze                       |             `1000–100000` |
| `l_y`      |              nessun limite | historical bias su `Y`                  |                     `0–5` |
| `l_m_y`    |              nessun limite | measurement bias su `Y`                 |                     `0–5` |
| `l_h_r`    |              nessun limite | historical bias su `R`                  |                     `0–5` |
| `l_h_q`    |              nessun limite | historical bias su `Q`                  |                     `0–5` |
| `l_m`      |              nessun limite | measurement bias/proxy di `R`           |                     `0–5` |
| `p_u`      |   nessun limite nel codice | quantità di `A=1` mantenuta             |                 **`0–1`** |
| `l_r`      |                    boolean | representation bias condizionato su `R` |              `False/True` |
| `l_o`      |                    boolean | omission bias                           |              `False/True` |
| `l_y_b`    |              nessun limite | interaction bias `R × A`                |                     `0–2` |
| `l_q`      |              nessun limite | influenza di `Q` su `Y`                 |                     `0–5` |
| `l_r_q`    |              nessun limite | influenza `R → Q`                       |                     `0–2` |
| `sy`       |             >0 consigliato | rumore di `Y`                           |                    `1–10` |
| `thr_supp` | tecnicamente nessun limite | soglia per eliminare proxy di `A`       |                 **`0–1`** |


# Generating historical biased dataset

This can be done with:
- `l_h_r`: Lambda coefficient for **historical bias** on R;
- `l_y`: Lambda coefficient for **historical bias** on the target y;

Let's choose to impact $Y$ directly.

In [ ]:
from src.spikes.BiasOnDemand.generate_dataset import create_synth

X_train, X_ind_train, X_supp_train, X_test, X_ind_test, X_supp_test, y_train, y_test, y_train_real, y_test_real = create_synth(dim=10000, l_y=10, l_m_y=0, thr_supp=1, l_h_r=0,  l_h_q=0, l_m=0, p_u=0, l_r=False, l_o=False, l_y_b=0, l_q=0, sy=0, l_r_q=0, l_m_y_non_linear=False, seed=420)

from pathlib import Path

path = "biased_dataset_histerical"

if path is None:
    path = 'my_new_dataset'
path = f'datasets/{path}'
# create directory if it does not exist yet
Path(path).mkdir(parents=True, exist_ok=True)

# save the generated dataset
X_train.to_csv(Path(path).joinpath('X_train.csv'))
X_ind_train.to_csv(Path(path).joinpath('X_ind_train.csv'))
X_supp_train.to_csv(Path(path).joinpath('X_supp_train.csv'))
X_test.to_csv(Path(path).joinpath('X_test.csv'))
X_ind_test.to_csv(Path(path).joinpath('X_ind_test.csv'))
X_supp_test.to_csv(Path(path).joinpath('X_supp_test.csv'))
y_train.to_csv(Path(path).joinpath('y_train.csv'))
y_test.to_csv(Path(path).joinpath('y_test.csv'))
y_train_real.to_csv(Path(path).joinpath('y_train_real.csv'))
y_test_real.to_csv(Path(path).joinpath('y_test_real.csv'))
X_train

In [ ]:
import biasondemand
import pandas as pd
#biasondemand.generate_dataset(path='biased_dataset_historical', dim=10000, l_y=2)
#biasondemand.generate_dataset(path='biased_dataset_historical', dim=10000, l_y=10, l_m=0, l_h_r=0, l_h_q=0 )
#biasondemand.generate_dataset(path='biased_dataset_historical', dim=1000, l_m=0.0, sy=0) # no bias


#biasondemand.generate_dataset(path='biased_dataset_historical', dim=10000, l_y=0, l_m_y=0, thr_supp=1, l_h_r=0,  l_h_q=0, l_m=0, p_u=0, l_r=False, l_o=False, l_y_b=0, l_q=0, sy=0, l_r_q=0, l_m_y_non_linear=False) # no bias (forced)


#biasondemand.generate_dataset(path='biased_dataset_historical', dim=10000, l_y=10, l_m_y=0, thr_supp=1, l_h_r=0,  l_h_q=0, l_m=0, p_u=0, l_r=False, l_o=False, l_y_b=0, l_q=0, sy=0, l_r_q=0, l_m_y_non_linear=False) # historical bias on Y

#biasondemand.generate_dataset(path='biased_dataset_historical', dim=1000, l_m=0.0, l_h_r=10, sy=0.0) # hist bias on R
#biasondemand.generate_dataset(path='biased_dataset_historical', dim=1000, l_m=0.0, l_h_q=10, sy=0.0) # hist bias on Q
#biasondemand.generate_dataset(path='biased_dataset_historical', dim=1000, l_m=0.0, l_y=10, sy=0.0) # hist bias on Y

dataset_name = 'biased_dataset_historical'

X_train = pd.read_csv(f'../spikes/datasets/{dataset_name}/X_train.csv', index_col=0)
y_train = pd.read_csv(f'../spikes/datasets/{dataset_name}/y_train.csv', index_col=0)
X_test = pd.read_csv(f'../spikes/datasets/{dataset_name}/X_test.csv', index_col=0)
y_test = pd.read_csv(f'../spikes/datasets/{dataset_name}/y_test.csv', index_col=0)

X = pd.concat([X_train, X_test], axis=0)
Y = pd.concat([y_train, y_test], axis=0)

pd.concat([X, Y], axis=1)

In [ ]:
from src.utils.data import preprocessing, label_encoding, sensitive_encoding

# R numerical, Q categorical, A sensitive
synth_data_info = {"biased_dataset_historical": {
    "preprocessing": {
      "target_encoding": {1: 1, 0: 0}, #FIXME: pay attention
      "pipeline": {
        "bool_pip" : {
          "imputer": {
            "params": {
              "strategy": "mean"
            }
          }
        },
        "num_pip" : {
          "imputer": {"params": {"strategy": "mean"}},
          "scaler": {"type":  "min_max", "params": {}}
        },
        "cat_pip": {
          "imputer": {"params": {"strategy": "most_frequent"}},
          "onehot": {"params":{"handle_unknown":"ignore", "sparse_output": False, "dtype":"np.int64"}}
        },
        "sens_pip": {}
      },
      "custom_mv": None
    },
    "fairness": {
      "sensitives": {
        "A": {
          1: 1, 0: 0 # FIXME: pay attention
        }
      },
      "fav_labels_mapping": {
        "1": 1,
        "0": 0
      }
    }
  }}

data_info = synth_data_info['biased_dataset_historical']

sensitives = list(data_info['fairness']['sensitives'].keys())  # names of the columns
sensitives_maps = data_info['fairness']['sensitives']  # mapping for multiple values subst. for sensitives
target_encoding = data_info['preprocessing']['target_encoding']

X['Q']=X['Q'].astype("object")

In [ ]:
X_prep = preprocessing(X, sensitives, data_info, dataset_name)
Y_prep = label_encoding(Y['Y'], map=target_encoding, verbose=False)
sensitive_encoding(X_prep, sensitives, sensitives_maps, verbose=False)

## Train unmitigated and mitigated

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from fairlearn.metrics import demographic_parity_difference
from fairlearn.postprocessing import ThresholdOptimizer
from sklearn import set_config
from src.utils.test import test_model
set_config(transform_output='pandas')
import numpy as np


X_train, X_probing, y_train, y_probing = train_test_split(X_prep, Y_prep, test_size=0.5, random_state=42, shuffle=True, stratify=Y_prep)
# index reset
# X_train, y_train --> for training un/mitigated model
X_train = X_train.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)
# X_probing, y_probing --> for probing the unmitigated model, label subst. etc...
X_probing = X_probing.reset_index(drop=True)
y_probing = y_probing.reset_index(drop=True)

X_train, X_test, y_train, y_test = train_test_split(X_train, y_train, test_size=0.3, random_state=42, shuffle=True, stratify=y_train)

# index reset
X_train = X_train.reset_index(drop=True)
X_test = X_test.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)

In [ ]:
unmitigated = LogisticRegression(random_state=42).fit(X_train, y_train.values.ravel())
unmitigated_preds = unmitigated.predict(X_test)

for s in sensitives:
    ss = X_test[s]
    accuracy_unmitigated, fairness_unmitigated = test_model(unmitigated, s, X_test, y_test, predictions_prob=False)

    print(f"accuracy: {accuracy_unmitigated['accuracy_score']} \n DP-diff: {fairness_unmitigated['DP-diff']}")

In [ ]:
mitigated = ThresholdOptimizer(
                   estimator=unmitigated,
                   constraints="demographic_parity",
                   objective="accuracy_score",
                   prefit=True,
                   predict_method='predict')

ss = sensitives[0]
#print("sensitive:", ss)
mitigated.fit(X_train, y_train, sensitive_features=X_train[ss])
mitigated_preds = mitigated.predict(X_test, sensitive_features=X_test[ss], random_state=42)

accuracy_mitigated, fairness_mitigated = test_model(mitigated, ss, X_test, y_test, predictions_prob=False)

print(f"accuracy: {accuracy_mitigated['accuracy_score']} \n DP-diff: {fairness_mitigated['DP-diff']}")

## Probing the mitigated model

In [ ]:
y_u = mitigated.predict(X_probing, sensitive_features=X_probing["A"], random_state=42)

In [ ]:
fav_unfav_mapping = data_info['fairness']['fav_labels_mapping']
print("\tFAV-UNFAV mapping", fav_unfav_mapping)
#print("\tFAV-UNFAV meaning", explain_fav_unfav_mapping(fav_unfav_mapping, target_encoding, cat=True))
y_u_encoded = label_encoding(pd.Series(y_u).astype(str), map=fav_unfav_mapping)
# ENCODE     y_probing
y_probing_encoded = label_encoding(pd.Series(y_probing).astype(str), map=fav_unfav_mapping)

## Label Generation and Preliminary Analysis of $D'$

In [ ]:
# NEW LABELS: Y'
y_prime = [1 if y_i_u != y_i else 0 for y_i_u, y_i in zip(y_u_encoded, y_probing_encoded)]

y_prime = np.array(y_prime)

# split new labels into groups
protected_mask = X_probing['A'] == 1
unprotected_mask = X_probing['A'] == 0

y_prime_protected = y_prime[protected_mask]
y_prime_unprotected = y_prime[unprotected_mask]

print("Stats of the corrected instances\n-------------------------------------")

print(f"Samples corrected: {(y_prime[y_prime == 1].shape[0] / y_prime.shape[0])*100} %")

protected_corrected = y_prime_protected[y_prime_protected == 1].shape[0]
print(f"Protected instances corrected: {(protected_corrected/y_prime[y_prime == 1].shape[0])*100} %")

unprotected_corrected = y_prime_unprotected[y_prime_unprotected == 1].shape[0]
print(f"Unprotected instances corrected: {(unprotected_corrected/y_prime[y_prime == 1].shape[0])*100} %")

# no. of protected samples
n_protected = X_probing[X_probing['A'] == 1].shape[0]
n_unprotected = X_probing[X_probing['A'] == 0].shape[0]

print(f"Correction per group:\n-----------------------------------\n protected:\t\t {(protected_corrected/n_protected)*100} %\n unprotected:\t {(unprotected_corrected/n_unprotected)*100} %")

import matplotlib.pyplot as plt
#!pip3 install seaborn
import seaborn as sns
sns.set(style='white')

correction_stats_df = pd.DataFrame(
    {
        "Group": ["Protected", "Unprotected"],
        #"Corrected": [protected_corrected, unprotected_corrected], #no of instances corrected per group
        #"Uncorrected ": [n_protected-protected_corrected, n_unprotected-unprotected_corrected], #no of instances uncorrected per group
        "Corrected": [protected_corrected/n_protected, unprotected_corrected/n_unprotected], #no of instances corrected per group
    }
)
#correction_stats_df.set_index('Group').plot(kind='bar', color=["#3498db", "#95a5a6"], legend=False)
#plt.title("Percentage of correction per group")


In [ ]:
def print_label_gen_stats(stats):
    print("Stats of the corrected instances\n-------------------------------------".upper())

    print(f"Samples corrected:\t {(stats['n_corrected']/stats['total_samples'])*100} %")
    print(f"> from protected:\t {(stats['protected_corrected']/stats['n_corrected'])*100} %")
    print(f"> from unprotected:\t {(stats['unprotected_corrected']/stats['n_corrected'])*100} %")
    print()
    print("Per group".upper())
    print(f"> protected:\t\t {stats["protected_corrected"]/stats['n_protected']*100} %")
    print(f"> unprotected:\t\t {stats["unprotected_corrected"]/stats['n_unprotected']*100} %")

    print()

def stats_of_label_gen(X, labels, s, verbose=False):
    stats = {}

    protected_mask = X[s] == 1
    unprotected_mask = X[s] == 0

    labels_protected = labels[protected_mask]
    labels_unprotected = labels[unprotected_mask]

    stats["total_samples"] = labels.shape[0]
    stats["n_protected"] = X[protected_mask].shape[0]
    stats["n_unprotected"] = X[unprotected_mask].shape[0]
    stats["n_corrected"] = labels[labels==1].shape[0]
    stats["protected_corrected"] = labels_protected[labels_protected==1].shape[0]
    stats["unprotected_corrected"] = labels_unprotected[labels_unprotected==1].shape[0]

    if verbose: print_label_gen_stats(stats)

    return stats

_ = stats_of_label_gen(X_probing, y_prime, "A", verbose=True)

## Discriminator training

In [ ]:
plt.close()

# SPLITTING FOR DISCR. TRAINING
print(">>> Splitting for discriminator training...")
from imblearn.over_sampling import RandomOverSampler
X_prob_train, X_prob_test, y_prime_train, y_prime_test = train_test_split(X_probing, y_prime,
                                                                          test_size=0.2,
                                                                          random_state=42, shuffle=True,
                                                                          stratify=y_prime)

X_prob_train = X_prob_train.reset_index(drop=True)
X_prob_test = X_prob_test.reset_index(drop=True)

ros = RandomOverSampler(random_state=42)  # inizializza l"oversampler
X_prob_train, y_prime_train = ros.fit_resample(X_prob_train, y_prime_train)

X_prob_train = X_prob_train.reset_index(drop=True)
X_prob_test = X_prob_test.reset_index(drop=True)

In [ ]:
# !pip3 install xgboost
from xgboost import XGBClassifier

discriminator = XGBClassifier()
discriminator.fit(X_prob_train, y_prime_train)


In [ ]:
accuracy_discriminator, fairness_discriminator = test_model(discriminator, "A", X_prob_test, y_prime_test, predictions_prob=False)

In [ ]:
from src.utils.test import print_metrics_table
print(accuracy_discriminator['accuracy_score'])
print(accuracy_discriminator['f1_score'])


## Knowledge Extraction

### KernelSHAP

In [ ]:
import shap
from shap.explainers import KernelExplainer


background = shap.sample(X_probing, 100)  # choose K samples (e.g., 50–500)
#X_sample = shap.sample(X_probing, 1000) # per non metterci una vita ora, #FIXME: poi togliere e spiegare tutto

def model_wrapper(X):
    # for MLP
    if not isinstance(X, pd.DataFrame):
        X = pd.DataFrame(X, columns=X_probing.columns)

    return discriminator.predict_proba(X)

explainer = KernelExplainer(
    model_wrapper,
    data=background,
    link="logit"
)

#shap_values = explainer.shap_values(X_sample, nsamples=1000, l1_reg="num_features(100)")
shap_values = explainer.shap_values(X_probing, nsamples=1000, l1_reg="num_features(100)")


In [ ]:
instance = 3
shap.plots.waterfall(shap.Explanation(values=shap_values[instance,:,1],
                                      base_values=explainer.expected_value[1],
                                      data=X_probing.iloc[instance,:],
                                      feature_names=X_probing.columns))

In [ ]:
print (f'length of SHAP values: {len(shap_values)}')
print (f'shape of each element: {shap_values[0]. shape}')
shap.initjs()
shap.force_plot(
    explainer.expected_value[1],
    shap_values[instance, :, 1],
    X_probing.iloc[instance, :],
    link="logit"
)

In [ ]:
import warnings
warnings.filterwarnings("ignore", message="The NumPy global RNG was seeded by calling `np.random.seed`")

_ = stats_of_label_gen(X_probing, y_prime, "A", verbose=True)


shap.summary_plot(shap_values[:, :, 1], X_probing)

In [ ]:
from xgboost import plot_tree

plot_tree(discriminator, tree_idx=1,  # Primo albero
    rankdir="TB",  # Direzione top-to-bottom
)  # rankdir="TB" per verticale

plt.show()



### TreeSHAP

In [ ]:
from shap import TreeExplainer
import shap


explainer = shap.TreeExplainer(discriminator)

shap_values = explainer.shap_values(X_probing)

shap.summary_plot(shap_values, X_probing)


In [ ]:
shap_values[3,:]

In [ ]:
instance = 99
shap.plots.waterfall(shap.Explanation(values=shap_values[instance,:],
                                      base_values=explainer.expected_value,
                                      data=X_probing.iloc[instance,:],
                                      feature_names=X_probing.columns))

In [ ]:
#per group

shap.summary_plot(
    shap_values[protected_mask, :],
    X_probing[protected_mask],
    show=False,
    #color=cmap,
    max_display=10,
    plot_type="bar"
)

In [ ]:
shap.summary_plot(
    shap_values[unprotected_mask, :],
    X_probing[unprotected_mask],
    show=False,
    #color=cmap,
    max_display=10,
    plot_type="bar"
)

In [ ]:
import shap
mask = np.ones_like(X_probing[s], dtype=bool)

idx = np.random.choice(np.where(mask)[0], size=200, replace=False)

shap_values = shap_values[idx, :]

expl = shap.Explanation(
    values=shap_values,
    base_values=explainer.expected_value,
    data=X_probing.iloc[idx],
    feature_names=X_probing.columns
)

shap.plots.heatmap(expl)


### LIME


In [ ]:

import lime
import lime.lime_tabular

# LIME has one explainer for all the models
explainer = lime.lime_tabular.LimeTabularExplainer(X_probing.values, feature_names=X_probing.columns.values.tolist(), verbose=True, mode='classification')

#FIXME: ma devo mettere X_test????'
j = 5
exp = explainer.explain_instance(X_probing.values[j], discriminator.predict_proba, num_features=6)

exp.save_to_file("lime_explanation.html")

### Decision tree

In [ ]:
from sklearn.tree import DecisionTreeClassifier

# Per la classificazione
discriminator_tree = DecisionTreeClassifier(
    criterion="gini",  # o "entropy" per l'entropia
    random_state=42,
    max_depth=2,
)

# Per la regressione
# model = DecisionTreeRegressor(criterion="squared_error", max_depth=5, ...)

discriminator_tree.fit(X_prob_train, y_prime_train)


In [ ]:
accuracy_discriminator_tree, fairness_discriminator_tree = test_model(discriminator_tree, "A", X_prob_test, y_prime_test, predictions_prob=False)
print(accuracy_discriminator_tree)

In [ ]:
from sklearn.tree import plot_tree
import matplotlib.pyplot as plt

# Supponiamo che 'model' sia il tuo albero decisionale già allenato
plt.figure(figsize=(20, 10))  # Imposta la dimensione della figura
plot_tree(
    discriminator_tree,  # Il tuo modello DecisionTreeClassifier/Regressor
    feature_names=X_train.columns,  # Nomi delle feature (se disponibili)
    class_names=["UNCORRECTED", "Corrected"],  # Nomi delle classi (per la classificazione)
    filled=True,  # Colora i nodi in base alla classe dominante
    rounded=True,  # Arrotonda gli angoli dei nodi
    fontsize=10,  # Dimensione del font
    proportion=True,  # Mostra la proporzione dei campioni invece del conteggio
)
plt.title("Albero Decisionale Allenato", fontsize=16)
plt.show()


# Playground

In [ ]:
#pd.concat([X_probing, pd.DataFrame(y_prime, columns=["y_prime"]), y_probing_encoded.rename("y_probing", inplace=True) ], axis=1)

In [ ]:
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt

tsne = TSNE(n_components=2, random_state=42, perplexity=30)
X_tsne = tsne.fit_transform(X_probing)


In [ ]:

# I divide the dataset into protected (A=1) and unprotected instances (A=0)



# then I color them according to their y_prime, to see in proportion the impact on these groups

In [ ]:
correction_stats_df.set_index("Group")#.sum().sum()
#correction_stats_df.pivot(index="Group", columns="Corrected", values="Norm")

In [ ]:
X_tsne_protected = X_tsne[protected_mask]
X_tsne_unprotected = X_tsne[unprotected_mask]

In [ ]:

from matplotlib.colors import ListedColormap

# Dati di esempio

# Mappa le label ai colori
label_to_color = {0: "grey", 1: "red"}
colors = [label_to_color[label] for label in y_prime_protected]

# Crea una colormap personalizzata
custom_cmap = ListedColormap(["grey", "red"])

In [ ]:
#y_prime_df = pd.DataFrame(y_prime, columns=["y_prime"])
plt.figure(figsize=(5, 5))
scatter = plt.scatter(
    X_tsne_protected.iloc[:, 0],  # Prima componente t-SNE
    X_tsne_protected.iloc[:, 1],  # Seconda componente t-SNE
    c=y_prime_protected,
    cmap=custom_cmap# Colora in base alle etichette
)

plt.title("t-SNE Visualization of probing set (protected)")
plt.legend(
    *scatter.legend_elements(),   # Estrai le label dalla scatter plot
    title="y_corrected"                # Titolo della legenda
)
plt.show()

In [ ]:

colors = [label_to_color[label] for label in y_prime_unprotected]

# Crea una colormap personalizzata
custom_cmap = ListedColormap(["grey", "red"])

plt.figure(figsize=(5, 5))
scatter = plt.scatter(
    X_tsne_unprotected.iloc[:, 0],  # Prima componente t-SNE
    X_tsne_unprotected.iloc[:, 1],  # Seconda componente t-SNE
    c=y_prime_unprotected,
    cmap=custom_cmap# Colora in base alle etichette
)

plt.title("t-SNE Visualization of probing set (unprotected)")
plt.legend(
    *scatter.legend_elements(),   # Estrai le label dalla scatter plot
    title="y_corrected"                # Titolo della legenda
)
plt.show()

In [ ]:
plt.figure(figsize=(5, 5))
scatter = plt.scatter(
    X_tsne.iloc[:, 0],  # Prima componente t-SNE
    X_tsne.iloc[:, 1],  # Seconda componente t-SNE
    c=y_prime,           # Colora in base alle etichette
)

plt.title("t-SNE Visualization of probing set")
plt.legend(
    *scatter.legend_elements(),   # Estrai le label dalla scatter plot
    title="y_corrected"                # Titolo della legenda
)
plt.show()

In [ ]:
from matplotlib.colors import ListedColormap

# Dati di esempio

# Mappa le label ai colori
label_to_color = {0: "red", 1: "blue"}
colors = [label_to_color[label] for label in y_prime_protected]

# Crea una colormap personalizzata
custom_cmap = ListedColormap(["red", "blue"])

In [ ]:
plt.figure(figsize=(5, 5))
scatter = plt.scatter(
    X_tsne.iloc[:, 0],  # Prima componente t-SNE
    X_tsne.iloc[:, 1],  # Seconda componente t-SNE
    c=X_probing["A"],           # Colora in base alle etichette
)

plt.title("t-SNE Visualization of probing set")
plt.legend(
    *scatter.legend_elements(),   # Estrai le label dalla scatter plot
    title="A"                # Titolo della legenda
)
plt.show()

# Synth Data Exps Playground

In [ ]:
from src.utils.data import generate_synth_dataset
from src.synth_exps import probing_sets_dict

dim = 50000

# no bias
_, _, _ = generate_synth_dataset(dim, {}, dataset_name="no_bias")
# historical bias on Y
_, _, _ = generate_synth_dataset(dim, {"l_y":10}, dataset_name="historical_biased_y", seed=42)
# measurement bias on Y
X, Y, _ = generate_synth_dataset(dim, {"l_m_y":10}, dataset_name="measurement_biased_y", seed=42)

# NOTE: remember to run preprocessing script

In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from fairlearn.postprocessing import ThresholdOptimizer
from fairlearn.preprocessing import PrototypeRepresentationLearner
from fairlearn.reductions import ExponentiatedGradient
import os
import json
import src.config as config
import pandas as pd
import pickle
from sklearn.model_selection import train_test_split
from time import time
import re
import numpy as np
import shap

from src.exps import train_best_model, preprocess_debiasing, train_best_fair_model_parallel, keys_of_dict, print_metrics_table, combined_score
from src.utils.test import test_model
from src.utils.data import label_encoding

from fairlearn.reductions import DemographicParity

import warnings
warnings.filterwarnings("ignore")


d = "historical_biased_y"
max_size = 10000

In [ ]:
def data_loading(d, max_size=None):
    f = open(config.datasets_info_path, "r")
    datasets_info = json.load(f)
    f.close()

    data_info = datasets_info[d]
    sensitives = list(data_info['fairness']['sensitives'].keys())  # names of the columns
    sensitives_maps = data_info['fairness']['sensitives']  # mapping for multiple values subst. for sensitives
    target_encoding = data_info['preprocessing']['target_encoding']

    padding = 2
    width = len(d) + padding * 2

    print("+" + "-" * width + "+")
    print("|" + d.center(width) + "|")
    print("+" + "-" * width + "+")

    print(">>> Dataset Info")
    print("\tsensitives:", sensitives)

    print("\t\t", sensitives_maps)
    print("\ttarget:", target_encoding)

    if os.path.exists(config.datasets_path + "/{}.preprocessed".format(d)):
        with open(config.datasets_path + "/{}.preprocessed".format(d), "rb") as f:
            X, Y = pickle.load(f)
        print("\tDataset loaded from disk.")
    else:
        raise Exception("Dataset not found. Run data_preprocess.py first.")

    print(">>> Shuffling & Splitting")

    if max_size is not None:
        print("\tmax_size:", max_size)
        X, _, Y, _ = train_test_split(
            X, Y,
            train_size=max_size,
            stratify=Y,
            random_state=42
        )
    return X, Y, data_info, sensitives, sensitives_maps, target_encoding

def unmitigated_training(unm_model, X_train, y_train, X_test, y_test):
    print(">>> (1) Unmitigated model ({}) training...".format(model_types[unm_model].__name__))
    unmitigated_model_path = "../out/models/unmitigated_{}_{}.model".format(unm_model, d)

    unmitigated_model_path = "../" + unmitigated_model_path #FIXME:


    # Search for best model
    unmitigated_halving_params = {
        "estimator": model_types[unm_model](random_state=42),
        "param_grid": model_grids[unm_model],
        "scoring": "f1_macro",
        "random_state": 42,
        "cv": 5,
        "factor": 2,
        "n_jobs": -1,
        "min_resources": int(len(X_train) / 2)
        # usa tutti i campioni nel fold per evitare problemi con classi uniche
    }
    unmitigated, unmitigated_params = train_best_model(unmitigated_halving_params, X_train, y_train,
                                                       unmitigated_model_path, verbose=True)
    unmitigated_accuracy_dict, unmitigated_fairness_dict = test_model(unmitigated, s, X_test, y_test)
    return unmitigated, unmitigated_params, unmitigated_accuracy_dict, unmitigated_fairness_dict

In [ ]:
# settings
unm_model_pool = ['lr']
debiasing_pool = ['to', 'eg', 'prl']
debiasing_prep = ['prl']
discr_model_pool = ['xgb']
test_size = 0.2

model_types = {
    "lr": LogisticRegression,
    "mlp": MLPClassifier,
    "to": ThresholdOptimizer,
    "xgb": XGBClassifier,
    "prl": PrototypeRepresentationLearner,
    "eg": ExponentiatedGradient
}
# Cols for exps results
# FIXME: da cambiare
res_cols = config.res_cols

preprocessing_debiasing = [PrototypeRepresentationLearner]

model_grids = config.model_grids
fairness_levels = config.fairness_lv

from_exp = 1
exp_no = 1
results_df = pd.DataFrame(columns=res_cols)
checkpoint_exists = False

# check if a checkpoint already exists
if os.path.exists(config.checkpoint_path.format(d)):
    checkpoint_exists = True
    results_df = pd.read_pickle(config.checkpoint_path.format(d))
    from_exp = len(results_df)
    print("Checkpoint containing {} tests loaded.".format(from_exp))

# load data
X, Y, data_info, sensitives, sensitives_maps, target_encoding = data_loading(d, max_size=max_size)

In [ ]:
def mitigated_training(debiasing, flv, s, d):
    print(">>> (2) Debiasing ({}) model on {}. Fairness lv. {}".format(model_types[debiasing].__name__, s, flv))
    mitigated_model_path = "/../out/models/mitigated_{}_{}_{}-{}.model".format(debiasing, flv, s, d)
    prep_path = "../../out/data/{}_{}_{}_{}".format(debiasing, flv, s, d) # FIXME:

    trans_model = None
    model_versions = None

    # add fairness lv to debiasing model grid
    for k, v in fairness_levels[flv][debiasing].items():
        model_grids[debiasing][k] = v

    if debiasing in debiasing_prep:

        # if preprocessing algo then
        # if preprocessing is cached then
        if os.path.exists(prep_path + ".data_versions") and os.path.exists(
                prep_path + ".data_versions_models"):

            with open(prep_path + ".data_versions", "rb") as f:
                data_versions = pickle.load(f)
            print("Data versions loaded from disk.")

            with open(prep_path + ".data_versions_models", "rb") as f:
                model_versions = pickle.load(f)
            print("Model versions loaded from disk.")
        else:
            data_versions, model_versions = preprocess_debiasing(model_types[debiasing],
                                                                 model_grids[debiasing], X_train, y_train,
                                                                 X_test, verbose=True,
                                                                 pickle_path="../../out/data/{}_{}_{}_{}.data_versions".format(
                                                                     debiasing, flv, s, d)) #FIXME: path

        pattern = re.compile(rf"{debiasing}_{flv}_{s}_{d}_best_model\((\d+)\)$")
        full_path = None

        #for filename in os.listdir("../out/models/"): #FIXME:
        for filename in os.listdir("../../out/models/"):
            match = pattern.match(filename)
            if match:
                best_index = int(match.group(1))
                trans_model = model_versions[best_index]
                print("Best data version", best_index)
                full_path = os.path.join("../out/models/", filename)

        if full_path is not None:
            #with open(full_path, "rb") as f:
            with open("../"+full_path, "rb") as f: #FIXME:
                mitigated = pickle.load(f)
                mitigated_params = mitigated.get_params()
            print("Best model loaded from disk.")
        else:
            lr_grid = {
                "penalty": ["l2"],
                "C": [0.1, 1.0],
                "solver": ["lbfgs"],
                "max_iter": [1000],
                "tol": [1e-4, 1e-2],
                "fit_intercept": [True],
                "class_weight": ["balanced", None],
            }

            best_search_score = -1
            best_search = None
            best_index = None

            for i, (X_train_transformed, X_test_transformed) in enumerate(data_versions):
                search_params = {
                    "estimator": LogisticRegression(),
                    "param_grid": lr_grid,
                    "X_train": pd.DataFrame(X_train_transformed),
                    "y_train": y_train,
                    "sensitive_col": s,
                    #"model_path": "../out/models/{}_{}_{}_{}_v{}".format(debiasing, flv, s, d, i), # FIXME:
                     "model_path": "../../out/models/{}_{}_{}_{}_v{}".format(debiasing, flv, s, d, i), # FIXME:
                    "custom_sensitives": X_train[s],
                }
                mit, mit_params, search = train_best_fair_model_parallel(**search_params)
                if search.best_score_ > best_search_score:
                    best_search = search
                    best_search_score = search.best_score_
                    best_index = i
                    trans_model = model_versions[best_index]

            mitigated = best_search.best_estimator_
            mitigated_params = best_search.best_params_

            # with open("../out/models/{}_{}_{}_{}_best_model({})".format(debiasing, flv, s, d, best_index) # FIXME
            with open("../../out/models/{}_{}_{}_{}_best_model({})".format(debiasing, flv, s, d, best_index), # FIXME
                      "wb") as f:
                pickle.dump(mitigated, f)
            print("Best model cached to disk.")

        mitigated_accuracy_dict, mitigated_fairness_dict = test_model(mitigated, s,
                                                                      data_versions[best_index][1], y_test,
                                                                      custom_sensitives=X_test[s])
    else:
        if debiasing == 'to':
            #for param_dict in model_grids[debiasing]:
            model_grids[debiasing]["estimator"] = [unmitigated]

        mitigated, mitigated_params, mitigated_search = train_best_fair_model_parallel(
            estimator=model_types[debiasing]() if debiasing == 'to' else model_types[debiasing](
                estimator=LogisticRegression(), constraints=DemographicParity()),
            param_grid=model_grids[debiasing],
            X_train=X_train,
            y_train=y_train,
            sensitive_col=s,
            #model_path=mitigated_model_path, #FIXME:
            model_path="../"+mitigated_model_path, # FIXME:
            cv=5,
            random_state=42,
            verbose=True
        )

        mitigated_accuracy_dict, mitigated_fairness_dict = test_model(mitigated, s, X_test, y_test)

    # for key in unmitigated_accuracy_dict.keys(): print(key,unmitigated_accuracy_dict[key], mitigated_accuracy_dict[key])
    # for key in unmitigated_fairness_dict.keys(): print(key,unmitigated_fairness_dict[key], mitigated_fairness_dict[key])
    print("\tDP-diff", unmitigated_fairness_dict['DP-diff'], "-->", mitigated_fairness_dict['DP-diff'])
    print("\tDP-ratio", unmitigated_fairness_dict['DP-ratio'], "-->", mitigated_fairness_dict['DP-ratio'])

    return mitigated, mitigated_params, mitigated_accuracy_dict, mitigated_fairness_dict, trans_model

In [ ]:
def model_probing(mitigated, X_probing, s, trans_model, debiasing):
    print(f">>> Probing mitigated {debiasing} model...")

    if isinstance(mitigated, ExponentiatedGradient):
        y_u = mitigated.predict(X_probing, random_state=42)
    elif debiasing == 'prl':
        if trans_model is None: raise ValueError("trans_model is None")
        #mitigated_params = [trans_model.get_params(), mitigated_params] #FIXME: non è un fixme, solo per note
        y_u = mitigated.predict(trans_model.transform(X_probing)) #FIXME: non è un fixme, solo per note
    else:
        y_u = mitigated.predict(X_probing, sensitive_features=X_probing[s], random_state=42)

        # y_u = mitigated.predict(X_probing, random_state=42) if isinstance(mitigated, ExponentiatedGradient)  else mitigated.predict(X_probing, sensitive_features=X_probing[s], random_state=42)
    return y_u

def label_generation(y_mitigated, y_probing):
    from src.exps import explain_fav_unfav_mapping
    # FAV/UNFAV MAPPING
    fav_unfav_mapping = data_info['fairness']['fav_labels_mapping']
    print("\tFAV-UNFAV mapping", fav_unfav_mapping)
    print("\tFAV-UNFAV meaning", explain_fav_unfav_mapping(fav_unfav_mapping, target_encoding, cat=True))

    # ENCODE y_mitigated
    y_u_encoded = label_encoding(pd.Series(y_mitigated).astype(str), map=fav_unfav_mapping)
    # ENCODE     y_probing
    y_probing_encoded = label_encoding(pd.Series(y_probing).astype(str), map=fav_unfav_mapping)
    # NEW LABELS: Y'
    y_prime = [1 if y_i_u != y_i else 0 for y_i_u, y_i in zip(y_u_encoded, y_probing_encoded)]

    return np.array(y_prime), y_u_encoded, y_probing_encoded

def print_label_gen_stats(stats):
    print("\nStats of the corrected instances\n-------------------------------------".upper())

    print(f"Samples corrected:\t {(stats['n_corrected']/stats['total_samples'])*100} %")
    print(f"> from protected:\t {(stats['protected_corrected']/stats['n_corrected'])*100} %")
    print(f"> from unprotected:\t {(stats['unprotected_corrected']/stats['n_corrected'])*100} %")
    print()
    print("Per group".upper())
    print(f"> protected:\t\t {stats["protected_corrected"]/stats['n_protected']*100} %")
    print(f"> unprotected:\t\t {stats["unprotected_corrected"]/stats['n_unprotected']*100} %")

    print()

def stats_of_label_gen(X, labels, s, verbose=False):
    stats = {}

    protected_mask = X[s] == 1
    unprotected_mask = X[s] == 0

    labels_protected = labels[protected_mask]
    labels_unprotected = labels[unprotected_mask]

    stats["total_samples"] = labels.shape[0]
    stats["n_protected"] = X[protected_mask].shape[0]
    stats["n_unprotected"] = X[unprotected_mask].shape[0]
    stats["n_corrected"] = labels[labels==1].shape[0]
    stats["protected_corrected"] = labels_protected[labels_protected==1].shape[0]
    stats["unprotected_corrected"] = labels_unprotected[labels_unprotected==1].shape[0]

    if verbose: print_label_gen_stats(stats)

    return stats

In [ ]:
def fix_new_dataset(X_train, y_train, mode="RandomOverSampling"):
    if mode == "RandomOverSampling":
        from imblearn.over_sampling import RandomOverSampler
        ros = RandomOverSampler(random_state=42)
        X_train, y_train = ros.fit_resample(X_train, y_train)
        X_train = X_train.reset_index(drop=True)
        #y_train = y_train.reset_index(drop=True)
    return X_train, y_train

def discriminator_training(X_train, y_train, X_test, y_test, discr_model, debiasing, flv, s, bias):
    print(">>> (3) Training discriminator ({}) on new labels...".format(
                            model_types[discr_model].__name__))
    #discr_model_path = "../out/models/discr_{}_{}_{}-{}-{}.model".format(discr_model, debiasing, flv, s, d)
    discr_model_path = "../../out/models/discr_{}_{}_{}-{}-{}_prob_{}.model".format(discr_model, debiasing, flv, s, d, bias) #FIXME:

    discriminator_halving_params = {
        "estimator": model_types[discr_model](),
        "param_grid": model_grids[discr_model],
        "scoring": "f1_macro",
        "random_state": 42,
        "cv": 5,
        "factor": 2,
        "n_jobs": -1,
        "min_resources": int(len(X_train) / 2)
    }

    discriminator, discriminator_params = train_best_model(discriminator_halving_params, X_train,
                                                           y_train, discr_model_path,
                                                           verbose=True)

    discr_accuracy_dict, discr_fairness_dict = test_model(discriminator, s, X_test, y_test)

    print_metrics_table(keys_of_dict(discr_accuracy_dict,
                                     ['accuracy', 'auc', 'f1_score', 'precision', 'recall',
                                      'matthew_coeff']))
    print_metrics_table(discr_fairness_dict)


    return discriminator, discriminator_params, discr_accuracy_dict, discr_fairness_dict

def get_knowledge_of_debiasing(discriminator, X_probing, explainer_path):
    background = None
    shap_values = None
    expected_value = None
    explainer = None

    if os.path.exists(explainer_path):
        with open(explainer_path, "rb") as f:
          artifact = pickle.load(f) # FIXME: avoid for efficiency purposes
        print("Shap values are already computed.", explainer_path)
    else:
        if isinstance(discriminator, XGBClassifier):
            # treeSHAP
            print(">>> TreeSHAP")
            explainer = shap.TreeExplainer(discriminator)
            shap_values = explainer.shap_values(X_probing)
            expected_value = explainer.expected_value
        else:
            background = shap.sample(X_probing, 100)  # also for preprocessing should be valid

            k = int(1.0 * X_probing.shape[1])

            def model_wrapper(X):
                # for MLP
                if not isinstance(X, pd.DataFrame):
                    X = pd.DataFrame(X, columns=X_probing.columns)

                return discriminator.predict_proba(X)

            explainer = KernelExplainer(
                model_wrapper,
                data=background,
                link="logit"
            )

            shap_values = explainer.shap_values(X_probing, nsamples=1000, l1_reg=f"num_features({k})")
            expected_value = explainer.expected_value

    # FIXME: when add new probing
    artifact = {
        "shap_values": shap_values,
        #"X": X_probing,
        "background": background,
        "expected_value": expected_value,
        "feature_names": X_probing.columns.tolist(),
        "explainer": explainer
    }

    with open(explainer_path, "wb") as f:
        pickle.dump(artifact, f)
    print(">>> Shap explainer saved to disk", explainer_path)
    return artifact


In [ ]:
# TODO: move up
probing_sets_dict = {
    "no_bias": {},
    "historically_biased_y": {"l_y":10},
    "measurement_biased_y": {"l_m_y":10},
    "historically_biased_r": {"l_h_r":10},
    "historically_biased_q": {"l_h_q":10},
    #"measurement_biased_r": {"l_m":10},
}

def get_probing_sets(probing_sets_dict, dim, d, verbose=True):
    from src.utils.data import generate_synth_dataset, sensitive_encoding


    probing_sets = {}

    if verbose: print(">>> Generating synthetic probing sets")
    for bias, params in probing_sets_dict.items():
        if verbose: print("\t"+bias)
        # FIXME: path
        X, Y, _ = generate_synth_dataset(dim, params, dataset_name=f"{d}_{bias}.probing", seed=42)
        # pre-process probing set (with the same preprocessor as d)
        # load d preprocessor
        with open(config.preprocessors_path.format(d), "rb") as f:
            dataset_preprocessor = pickle.load(f)
            # FIXME: reorder columns
        if "bias" in d:
            X['Q'] = X['Q'].astype("object")
            sensitives_maps['A'] = {1: 1, 0: 0}
        # preprocess X
        X = dataset_preprocessor.transform(X)
        # encode sensitive and labels
        #FIXME: encoding
        Y=label_encoding(Y, map={1: 1, 0: 0}, verbose=False) # synth
        sensitive_encoding(X, sensitives, sensitives_maps, verbose=False)
        probing_sets[bias]= (X, Y)

    return probing_sets

In [ ]:
"""for s in sensitives:
    X_train, X_test, y_train, y_test = train_test_split(X_train_unm, y_train_unm, test_size=test_size,
                                                        random_state=42, shuffle=True, stratify=y_train_unm) #FIXME: change for new probing

    # index reset
    X_train = X_train.reset_index(drop=True)
    X_test = X_test.reset_index(drop=True)
    y_train = y_train.reset_index(drop=True)
    y_test = y_test.reset_index(drop=True)

    for unm_model in unm_model_pool:
        start_time_unmitigated = time()
        unmitigated, unmitigated_params, unmitigated_accuracy_dict, unmitigated_fairness_dict = unmitigated_training(unm_model, X_train, y_train, X_test, y_test)
        end_time_unmitigated = time()

        for flv in fairness_levels.keys():
            for debiasing in debiasing_pool:
                start_time_debiasing = time()
                mitigated, mitigated_params, mitigated_accuracy_dict, mitigated_fairness_dict, trans_model = mitigated_training(debiasing, flv, s, d)
                end_time_debiasing = time()
                # MODEL PROBING
                # TODO: NEW MODEL PROBING
                y_mitigated = model_probing(mitigated, X_probing, s, trans_model, debiasing)
                # LABEL GENERATION
                y_prime, y_u_encoded, y_probing_encoded = label_generation(y_mitigated, y_probing)
                # STATS OF D'
                new_dataset_stats = stats_of_label_gen(X_probing, y_prime, s, verbose=True)
                # APPLY OVERSAMPLING TO D'
                X_prob_train, X_prob_test, y_prime_train, y_prime_test = train_test_split(X_probing, y_prime, test_size=test_size, random_state=42, shuffle=True, stratify=y_prime)
                X_prob_train = X_prob_train.reset_index(drop=True)
                X_prob_test = X_prob_test.reset_index(drop=True)
                X_prob_train_os, y_prime_train_os = fix_new_dataset(X_prob_train, y_prime_train)
                # DISCRIMINATOR TRAINING
                for discr_model in discr_model_pool:
                    if checkpoint_exists:
                        if exp_no <= from_exp:
                            # FIXME: add check
                            exp_no += 1
                            continue
                    start_time_discriminator = time()
                    discriminator, discriminator_params, discr_accuracy_dict, discr_fairness_dict = discriminator_training(X_prob_train_os, y_prime_train_os, X_prob_test, y_prime_test, discr_model, debiasing, flv, s)
                    end_time_discriminator = time()

                    # GETTINKG KNOWLEDGE ABOUT DEBIASING
                    #explainer_path = "../out/models/shap_{}_{}-{}-{}-{}-{}.model".format(discr_model, unm_model, debiasing, flv, s, d)
                    explainer_path = "../../out/models/shap_{}_{}-{}-{}-{}-{}.model".format(discr_model, unm_model, debiasing, flv,
                                                                  s, d) #FIXME:
                    artifacts = get_knowledge_of_debiasing(discriminator, X_probing, explainer_path)

                    end_time = time()
                    #TODO: refactoring results # FIXME:
                    results_df.loc[exp_no] = {
                            "total_time": end_time - start_time_unmitigated,  # for one whole run of exp #
                            # Data Info
                            "dataset": d,
                            "sensitive": s,
                            "test_size": test_size,
                            # Unmitigated Model
                            "unm_model_type": model_types[unm_model].__name__,
                            "unm_model_accuracy": unmitigated_accuracy_dict,  # JSON with objs
                            "unm_model_fairness": unmitigated_fairness_dict,  # JSON with objs
                            "unm_model_params": unmitigated_params,
                            "unm_model_exp_time": end_time_unmitigated - start_time_unmitigated,
                            # Mitigated Model (Debiasing)
                            "debiasing_lv": flv,
                            "debiasing_type": model_types[debiasing].__name__,
                            "debiasing_accuracy": mitigated_accuracy_dict,  # JSON with objs
                            "debiasing_fairness": mitigated_fairness_dict,  # JSON with objs
                            "debiasing_params": mitigated_params,
                            "debiasing_combined_score": combined_score(mitigated_accuracy_dict, mitigated_fairness_dict),
                            "debiasing_exp_time": end_time_debiasing - start_time_debiasing,
                            # Discriminator Model
                            "discr_model_type": model_types[discr_model].__name__,
                            "discr_model_accuracy": discr_accuracy_dict,  # JSON with objs
                            "discr_model_fairness": discr_fairness_dict,  # JSON with objs,
                            "discr_model_params": discriminator_params,
                            "discr_model_exp_time": end_time_discriminator - start_time_discriminator,
                            "y_u": y_u_encoded,  # predictions of discriminator
                            "y_probing": y_probing_encoded,  # ground_truths
                            "y_prime": y_prime  # new labels
                            # XAI
                        }
                    exp_no += 1
                    #results_df.to_pickle(config.checkpoint_path.format(d))
                    results_df.to_pickle("../"+config.checkpoint_path.format(d)) #FIXME:
    # save results to disk and delete checkpoints
    # results_df.to_pickle(config.results_path.format(d))
    results_df.to_pickle("../"+config.results_path.format(d))
    #if os.path.exists(config.checkpoint_path.format(d)): os.remove(config.checkpoint_path.format(d))
    if os.path.exists("../"+config.checkpoint_path.format(d)): os.remove("../"+config.checkpoint_path.format(d)) # FIXME:
"""

In [ ]:
for s in sensitives:
    X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=test_size, random_state=42, shuffle=True, stratify=Y)

    # index reset
    X_train = X_train.reset_index(drop=True)
    X_test = X_test.reset_index(drop=True)
    y_train = y_train.reset_index(drop=True)
    y_test = y_test.reset_index(drop=True)

    for unm_model in unm_model_pool:
        start_time_unmitigated = time()
        unmitigated, unmitigated_params, unmitigated_accuracy_dict, unmitigated_fairness_dict = unmitigated_training(unm_model, X_train, y_train, X_test, y_test)
        end_time_unmitigated = time()

        for flv in fairness_levels.keys():
            for debiasing in debiasing_pool:
                start_time_debiasing = time()
                mitigated, mitigated_params, mitigated_accuracy_dict, mitigated_fairness_dict, trans_model = mitigated_training(debiasing, flv, s, d)
                end_time_debiasing = time()
                # get probing sets
                probing_sets = get_probing_sets(probing_sets_dict, 5000, d)
                for bias in probing_sets.keys():
                    print("PROBING SET: ", bias)
                    X_probing, y_probing = probing_sets[bias]
                    X_probing = X_probing[X.columns] # force columns to be in the same order
                    y_mitigated = model_probing(mitigated, X_probing, s, trans_model, debiasing)
                    # LABEL GENERATION
                    y_prime, y_u_encoded, y_probing_encoded = label_generation(y_mitigated, y_probing)
                    # STATS OF D'
                    new_dataset_stats = stats_of_label_gen(X_probing, y_prime, s, verbose=True)
                    # APPLY OVERSAMPLING TO D'
                    X_prob_train, X_prob_test, y_prime_train, y_prime_test = train_test_split(X_probing, y_prime, test_size=test_size, random_state=42, shuffle=True, stratify=y_prime)
                    X_prob_train = X_prob_train.reset_index(drop=True)
                    X_prob_test = X_prob_test.reset_index(drop=True)
                    X_prob_train_os, y_prime_train_os = fix_new_dataset(X_prob_train, y_prime_train)

                    # DISCRIMINATOR TRAINING
                    for discr_model in discr_model_pool:
                        if checkpoint_exists:
                            if exp_no <= from_exp:
                                # FIXME: add check
                                exp_no += 1
                                continue
                        start_time_discriminator = time()
                        discriminator, discriminator_params, discr_accuracy_dict, discr_fairness_dict = discriminator_training(X_prob_train_os, y_prime_train_os, X_prob_test, y_prime_test, discr_model, debiasing, flv, s, bias)
                        end_time_discriminator = time()

                        # GETTINKG KNOWLEDGE ABOUT DEBIASING
                        #explainer_path = "../out/models/shap_{}_{}-{}-{}-{}-{}.model".format(discr_model, unm_model, debiasing, flv, s, d)
                        explainer_path = f"../../out/models/shap_{discr_model}_{unm_model}-{debiasing}-{flv}-{s}-{d}_prob_{bias}.model" #FIXME:

                        artifacts = get_knowledge_of_debiasing(discriminator, X_probing, explainer_path)

                        end_time = time()
                        # RESULTS
                        results_df.loc[exp_no] = {
                                "total_time": end_time - start_time_unmitigated,  # for one whole run of exp #
                                # Data Info
                                "dataset": d,
                                "sensitive": s,
                                "test_size": test_size,
                                "probing_set": bias,
                                # Unmitigated Model
                                "unm_model_type": model_types[unm_model].__name__,
                                "unm_model_accuracy": unmitigated_accuracy_dict,  # JSON with objs
                                "unm_model_fairness": unmitigated_fairness_dict,  # JSON with objs
                                "unm_model_params": unmitigated_params,
                                "unm_model_exp_time": end_time_unmitigated - start_time_unmitigated,
                                # Mitigated Model (Debiasing)
                                "debiasing_lv": flv,
                                "debiasing_type": model_types[debiasing].__name__,
                                "debiasing_accuracy": mitigated_accuracy_dict,  # JSON with objs
                                "debiasing_fairness": mitigated_fairness_dict,  # JSON with objs
                                "debiasing_params": mitigated_params,
                                "debiasing_combined_score": combined_score(mitigated_accuracy_dict, mitigated_fairness_dict),
                                "debiasing_exp_time": end_time_debiasing - start_time_debiasing,
                                # Discriminator Model
                                "discr_model_type": model_types[discr_model].__name__,
                                "discr_model_accuracy": discr_accuracy_dict,  # JSON with objs
                                "discr_model_fairness": discr_fairness_dict,  # JSON with objs,
                                "discr_model_params": discriminator_params,
                                "discr_model_exp_time": end_time_discriminator - start_time_discriminator,
                                "y_u": y_u_encoded,  # predictions of discriminator
                                "y_probing": y_probing_encoded,  # ground_truths
                                "y_prime": y_prime,  # new labels
                                # XAI
                                "new_dataset_stats": new_dataset_stats
                            }
                        exp_no += 1
                        #results_df.to_pickle(config.checkpoint_path.format(d))
                        results_df.to_pickle("../"+config.checkpoint_path.format(d)) #FIXME:
    # save results to disk and delete checkpoints
    # results_df.to_pickle(config.results_path.format(d))
    results_df.to_pickle("../"+config.results_path.format(d))
    #if os.path.exists(config.checkpoint_path.format(d)): os.remove(config.checkpoint_path.format(d))
    if os.path.exists("../"+config.checkpoint_path.format(d)): os.remove("../"+config.checkpoint_path.format(d)) # FIXME:

In [ ]:
import pandas as pd
import src.config as config
d = "historical_biased_y"
pd.read_pickle("../"+config.results_path.format(d))[:5]#['y_u'][1]

In [ ]:
!pwd

Note:
- measurement and historical (both on y) biases give the same stats of D'
- for measurement biased l_m there is the problem of the missing column R
- with different seeds, it seems that probing is failing to detect feature importance

In [ ]:
# Viz. SHAP
import pickle
import shap
import src.config as config
import warnings
import pandas as pd
warnings.filterwarnings("ignore", category=FutureWarning)

d = "historical_biased_y"
bias = "no_bias"
s = "A"
flv = "high"
debiasing = "to"
discr = "xgb"

def get_shap_things(d = "historical_biased_y", bias = "no_bias", s = "A", flv = "high", debiasing = "prl", discr = "xgb", verbose=True):

    probing_set_path = f"../../out/data/{d}_probing-set-{bias}.preprocessed"
    shap_path = f"../../out/models/shap_{discr}_lr-{debiasing}-{flv}-{s}-{d}_prob_{bias}.model"
    discriminator_path = f"../../out/models/discr_{discr}_{debiasing}_{flv}-{s}-{d}_prob_{bias}.model_seed42"

    if verbose:
        print(shap_path)

    # load probing set and then shap values
    X_probing, Y_probing = pd.read_pickle(probing_set_path)

    with open(shap_path, "rb") as file:
        artifacts = pickle.load(file)

    #with open(discriminator_path, "rb") as file: discriminator = pickle.load(file)
    return artifacts, X_probing#, discriminator


In [ ]:
artifacts['shap_values'].shape

In [ ]:
artifacts, X_probing, discriminator = get_shap_things(bias="no_bias")
# [:, :, 1]
shap.summary_plot(artifacts['shap_values'], X_probing)

In [ ]:
artifacts, X_probing, _ = get_shap_things(d = "historical_biased_y", bias = "historically_biased_y", s = "A", flv = "high", debiasing = "prl", discr = "xgb")
shap.summary_plot(artifacts['shap_values'], X_probing)

In [ ]:
artifacts['shap_values'].mean(axis=0)

https://shap.readthedocs.io/en/latest/generated/shap.TreeExplainer.html#shap.TreeExplainer.__init__
If “probability”, then we explain the output of the model transformed into probability space (note that this means the SHAP values now sum to the probability output of the model).
Currently the “probability” and “log_loss” options are only supported when feature_perturbation="interventional".


# Generazione dati sintetici


In [ ]:
from src.utils.data import generate_synth_dataset
from src.synth_exps import probing_sets_dict

dim = 50000

"""# no bias
_, _, _ = generate_synth_dataset(dim, {}, dataset_name="no_bias")
# historical bias on Y
_, _, _ = generate_synth_dataset(dim, {"l_y":10}, dataset_name="historical_biased_y", seed=42)
# measurement bias on Y
X, Y, _ = generate_synth_dataset(dim, {"l_m_y":10}, dataset_name="measurement_biased_y", seed=42)"""

for bias in probing_sets_dict.keys():
    _, _, _ = generate_synth_dataset(dim, probing_sets_dict[bias], dataset_name=bias, seed=42)


# HeatMAP

- righe $\rightarrow$ features;
- colonne $\rightarrow$ bias (probing_set)
- celle $\rightarrow$ valore di SHAP per la feature $i$ e il bias $j$

> Nota: le SV danno l'importanza della feature rispetto alla classe positiva (quindi quella corrected), perciò noi otteniamo l'importanza media (rispetto alla classe corrected) che una feature ha in un probing set con un certo bias.

In un certo senso dovremmo capire che il debiasing corregge un tipo di bias se effettivamente il pattern di "attivazione" coincide con il tipo di bias. Ad esempio se abbiamo fatto un bias su Q, le variabili Q dovrebbero attivarsi.

In [ ]:
def save_plot(plt, path, formats=None):
    if formats is None: formats = ["png", "pdf", "eps"]

    for fmt in formats:
        plt.savefig(f"{path}.{fmt}", dpi=300 if fmt == "png" else None, bbox_inches="tight")

import seaborn as sns
import pandas as pd
import src.config as config
import matplotlib.pyplot as plt
from src.config import probing_sets_dict

d= list(probing_sets_dict.keys())[3]
print(probing_sets_dict.keys())
s = "A"


results = pd.read_pickle("../"+config.results_path.format(d))

dict_prova = {
    "to": 'ThresholdOptimizer',
    "eg":'ExponentiatedGradient',
 "prl": 'PrototypeRepresentationLearner'
}
biases = list(probing_sets_dict.keys())
features = list(pd.read_pickle(f"../../rsc/datasets/{d}.preprocessed")[0].columns)


flvs = ["low", "medium", "high"]
debiasings = ['to']#["to", "eg", "prl"]

discr = "xgb"

no_bias_effect = None
for flv in flvs:
    for deb in debiasings:
        hm_df = pd.DataFrame(index=features)
        for b in biases:
            artifacts, probing_set = get_shap_things(d = d, bias = b, s = s, flv = flv, debiasing = deb, discr = discr, verbose = False)
            shap_values = pd.DataFrame(artifacts['shap_values'], columns=features)

            y_prime = pd.DataFrame(results[(results['debiasing_type'] == dict_prova[deb]) & (results['probing_set'] == b) & (results['debiasing_lv'] == flv)].iloc[0]['y_prime'])
            if y_prime.shape[1] > 1: raise ValueError("shape > 1")
            corrected_instances_mask =  (y_prime == 1)
            shap_values["corrected"] = corrected_instances_mask
            shap_values = shap_values[shap_values['corrected']==True].drop(["corrected"], axis=1)

            if b == "no_bias":
                no_bias_effect = shap_values.copy()
            #shap_values = shap_values - no_bias_effect

            #shap_values = shap_values.abs()
            shap_values = shap_values / np.abs(shap_values).max()
            mean_shap_values = shap_values.mean(axis=0).to_frame(name=b)
            hm_df = pd.concat([hm_df, mean_shap_values], axis=1)
            #hm_df = hm_df.abs()

        sns.heatmap(hm_df.transpose(), annot=True, fmt=".1f", vmin=-1,vmax=1, cmap="crest")
        plt.title(f"{d}-{deb}-{flv}-{s}")
        plt.ylabel("Probing set / Bias type")
        plt.xlabel("Feature")
        plt.tight_layout()
        plt.show()
        #break
    #break

In [ ]:
group = 1
for flv in flvs:
    for deb in debiasings:
        hm_df = pd.DataFrame(index=features)
        for b in biases:
            artifacts, probing_set = get_shap_things(d = d, bias = b, s = s, flv = flv, debiasing = deb, discr = discr, verbose = False)
            protected_mask = probing_set[s] == group
            shap_values = pd.DataFrame(artifacts['shap_values'], columns=features)


            y_prime = pd.DataFrame(results[(results['debiasing_type'] == dict_prova[deb]) & (results['probing_set'] == b) & (results['debiasing_lv'] == flv)].iloc[0]['y_prime'])
            if y_prime.shape[1] > 1: raise ValueError("shape > 1")
            corrected_instances_mask =  (y_prime == 1)
            shap_values["corrected"] = corrected_instances_mask
            shap_values = shap_values[shap_values['corrected']==True].drop(["corrected"], axis=1)

            shap_values = pd.DataFrame(artifacts['shap_values'][protected_mask], columns=features)
            if b == "no_bias":
                no_bias_effect = shap_values.copy()
            #shap_values = shap_values - no_bias_effect

            #shap_values = shap_values.abs()
            shap_values = shap_values / np.abs(shap_values).max()
            mean_shap_values = shap_values.mean(axis=0).to_frame(name=b)
            hm_df = pd.concat([hm_df, mean_shap_values], axis=1)
            #hm_df = hm_df.abs()

        sns.heatmap(hm_df.transpose(), annot=True, fmt=".1f", vmin=-1,vmax=1, cmap="crest")
        plt.title(f"{d}-{deb}-{flv}-{s} - Group {'Protected' if group == 1 else 'Unprotected'}")
        plt.ylabel("Probing set / Bias type")
        plt.xlabel("Feature")
        plt.tight_layout()
        plt.show()

In [ ]:
from src.synth_exps import print_label_gen_stats

for flv in flvs:
    for deb in debiasings:
        for b in biases:
            print(b, deb, flv)
            stats = pd.DataFrame(results[(results['debiasing_type'] == dict_prova[deb]) & (results['probing_set'] == b) & (results['debiasing_lv'] == flv)]).iloc[0]['new_dataset_stats']
            print_label_gen_stats(stats)

In [ ]:
plt.close()

# Viz stats of D'
- Correction % (per debiasing and flv): $n_{corrected} \over n$

In [ ]:
from itertools import product

def explode_metrics(d):
    return {key: d.get(key) for key in d.keys()}

def extract_metrics(df, metrics_to_explode, cols_to_explode_prefx):
    for col, metric in product(cols_to_explode_prefx, metrics_to_explode):
        column = "{}_{}".format(col, metric)
        extraction = df[column].apply(explode_metrics).apply(pd.Series).add_prefix(col+"_")
        df = pd.concat([df, extraction], axis=1)
    return df

import pandas as pd
import src.config as config

results = pd.read_pickle("../"+config.results_path.format(d))

In [ ]:

metrics_to_explode = ["stats"]
cols_to_explode_prefx = ["new_dataset"]

cols = [#'dataset', 'sensitive',
        'probing_set', 'debiasing_type', 'debiasing_lv', 'new_dataset_total_samples',
       'new_dataset_n_protected', 'new_dataset_n_unprotected',
       'new_dataset_n_corrected', 'new_dataset_protected_corrected',
       'new_dataset_unprotected_corrected']

pr = extract_metrics(results, metrics_to_explode, cols_to_explode_prefx)[cols]
pr = pr[pr["probing_set"] == "historically_biased_y"]
pr


In [ ]:
sns.barplot(
    x="debiasing_type",        # Variabile categoriale per l'asse x
    y="new_dataset_n_corrected", # Variabile numerica per l'asse y
    hue="debiasing_lv",      # Variabile per il raggruppamento (colore delle barre)
    data=pr,      # Dataset
    #estimator="mean", # Statistica da visualizzare (default: media)
    #ci=None,        # Disabilita l'intervallo di confidenza
    palette="pastel" # Palette di colori
)

In [ ]:
# TODO: derive % stats by applying lambda functs
# TODO: general graphs
pr.drop(["new_dataset_total_samples", "new_dataset_n_protected", "new_dataset_n_unprotected"], axis=1)[pr["debiasing_type"] =="PrototypeRepresentationLearner"].set_index(["debiasing_type", "debiasing_lv"]).plot(kind='bar', color=["blue", "red", "green"])
# TODO: per group maybe with pd

In [ ]:
# y_u -> y_mitigated
# y_probing -> "real"
# y_prime -> corrected
# UNFAV --> 0
# FAV --> 1
# TODO: graphs of UNFAV --> FAV
def instances_unfav_fav(y_mitigated, y_probing, mode="unfav_fav"):
    # Note: ensure both have the same indices
    c_mitigated = 1; c_probing = 0
    if mode == "fav_unfav": c_mitigated = 0; c_probing = 1
    if y_mitigated.shape != y_probing.shape: raise ValueError("Shapes do not match.")
    return (y_mitigated == c_mitigated) & (y_probing == c_probing)


In [ ]:
y_mitigated = results.iloc[0]['y_u'].reset_index(drop=True)
y_probing = results.iloc[0]['y_probing'].reset_index(drop=True)

instances_unfav_fav(y_mitigated, y_probing).sum()

# TODO: stessa cosa per group

**Note**:
- correzione sempre uguale (anche tra vari probing set), nella visualizzazione perché poi i count son diversi??
- dalle heatmap non si riesce a capire il tipo di bias. Overfitting?

Future works:
- [ ] generare probing in maniera dichiarativa
- [ ] spiegare debiasing ad-hoc, model-specific, auditing in real time

**Paper**:
- LR su auditing;
- schema grafico di motivation per capire dove sta il problema;
- diagramma framework stile comics/sketched;
- vignetta comica come motivazione;
- un titolo tipo post-mitigation black-box debiasing auditing using synthetic biased datasets

# Da fare
- [X] ~~più dataset biased (LEVARE R, USARE P_R, generazione controllata + pre-processing + file config datasets)~~
- [X] ~~controllare seed dei modelli mitigated, o se volendo si può fare anche nella search custom del train fair model~~ Si può fare anche se è custom la search, viene usato nello split del KFold. In più ogni modello ha possibilità di cambiare seed (costruttore o predict).
- [X] ~~controllare params debiasing in config~~, **controllare anche sul server**
- [X] ~~più mitigated con diversi seed nella search, faccio probing degli n mitigated, etichetta di corrected anche se solo uno dei mitigated dice che è corretto (vedere appunti sotto)~~
- [X] ~~costruire matrici VAR~~
- [X] refactoring (aggiungere n al caching dei modelli, far venire fuori dei viz carini e sistematici);
- [ ] fare prove con più istanze del dataset di partenza, es. [100, 500, 1000, 5000, 10000];
- [X] calcolare varianza dove si può, sputare fuori dei frame che indicano se il test è stato passato o meno;
- [X] **controllare se no_bias == y_real**: in realtà lo è ma per tutti i bias anche;
- [X] Viz solo dei corrected
- [ ] abilitare Oversampling?
- [ ] (xgboost TreeApproximator) https://github.com/rupensa/xccshap_ds2024/tree/master/src/XGBoostTreeApproximator
- [ ] esperimenti dati reali poi, aprire dataset e classificarli come R, A, Q ecc

## Debiasing
In pratica, allenare $n$ mitigated (es. n=10) con seed diversi e fargli probing. Una volta ottenute le $n$ predizioni, fare label generation in questo modo: se per l'istanza $i$ tutte le $n$ predizioni concordano (sia che siano $0$, sia che siano $1$) allora mettere $0$ nella predizione, altrimenti, anche se un solo mitigated ha dato una predizione diversa, mettere $1$. In questo modo creiamo $y^{var}$.
Poi usare lo stesso (in termini di parametri) $D$ che si è usato per la procedura AID e allenarlo su il probing set con $y^{var}$.
Dopodiché, per ogni variabile e setting (flv, debiasing_type, bias/probing) si fanno uscire dei frame come l'esempio riportato di seguito:

| F  AID              | F - VAR             |
|---------------------|---------------------|
| shap value F in AID | Shap value F in Var |

dove $F=\{A, Q, P\}$.

Infine fare un test statistico e se p-value << 0.05 allora ok, altrimenti AID non riesce a beccare quello che vogliamo fare.

Questione seed invece nei debiasing in sé

| [TO](https://fairlearn.org/main/api_reference/generated/fairlearn.postprocessing.ThresholdOptimizer.html#fairlearn.postprocessing.ThresholdOptimizer)                      | [EG](https://fairlearn.org/main/api_reference/generated/fairlearn.reductions.ExponentiatedGradient.html#fairlearn.reductions.ExponentiatedGradient)                        | [PRL](https://fairlearn.org/main/api_reference/generated/fairlearn.preprocessing.PrototypeRepresentationLearner.html#fairlearn.preprocessing.PrototypeRepresentationLearner)                       |
|-----------------------------|-------------------------------|-------------------------------|
| seed presente nella `predict` | seed presente nella `predict` | seed presente nel costruttore |

In [ ]:
from src.utils.data import generate_synth_dataset
from config import probing_sets_dict as bias_dict

x, y, y_real = generate_synth_dataset(100, {"l_m":1, "l_y":100, "l_y_q":100}, dataset_name="dummy")


In [ ]:
(y == y_real).sum()

In [ ]:
(y == y_real)

In [ ]:
ch_path = "../../out/results_historically_biased_q.df"
df = pd.read_pickle(ch_path)
#df[df["debiasing_type"] == "PrototypeRepresentationLearner"]
df[["debiasing_type", "debiasing_lv", "probing_set",  "discr_model_accuracy", "discriminator_var_accuracy"]]

In [ ]:
# TODO: calcolare stat su varianza
# ottenere per ogni feature (o anche complessiva?) una stat

In [ ]:
discr = "xgb"
debiasing = "to"
flv = "low"
s = "A"
bias = "historically_biased_y"

shap_path = f"../../out/models/shap_{discr}_lr-{debiasing}-{flv}-{s}-{d}_prob_{bias}.model"
var_shap_path =f"../../out/models/var_shap_{discr}_lr-{debiasing}-{flv}-{s}-{d}_prob_{bias}.model"

with open(shap_path, "rb") as f: shap_aid = pickle.load(f)
with open(var_shap_path, "rb") as f: shap_var = pickle.load(f)

shap_var_values = pd.DataFrame(shap_var['shap_values'], columns=shap_var['feature_names'])
shap_aid_values = pd.DataFrame(shap_aid['shap_values'], columns=shap_aid['feature_names'])

from scipy.stats import spearmanr, pearsonr, wilcoxon
from scipy.stats import ttest_rel
import numpy as np

test_df = pd.DataFrame(index=shap_var_values.columns)
for f in shap_var_values.columns:
    #pair = pd.DataFrame([shap_var_values[f].rename(f"{f}-VAR"), shap_aid_values[f].rename(f"{f}-AID")]).T
    aid = shap_aid_values[f].to_numpy()
    var = shap_var_values[f].to_numpy()
    r_pearson, p_pearson = pearsonr(aid, var)
    r_spearman, p_spearman = spearmanr(aid, var)
    test = wilcoxon(aid, var, alternative="two-sided", method="auto")
    w_stat, p_wilcoxon = test.statistic, test.pvalue
    t_stat, p_ttest = ttest_rel(aid, var)

    test_df.loc[f, "t_stat"] = t_stat
    test_df.loc[f, "p_ttest"] = p_ttest
    test_df.loc[f, 'p_wilcoxon'] = p_wilcoxon
    test_df.loc[f, 'w_stat'] = w_stat
    test_df.loc[f, "r_pearson"] = r_pearson
    test_df.loc[f, 'p_pearson'] = p_pearson
    test_df.loc[f, 'r_spearman'] = r_spearman
    test_df.loc[f, 'p_spearman'] = p_spearman

test_df

In [ ]:
from statsmodels.stats.multitest import multipletests
# Correzione Benjamini–Hochberg, separata per ciascun gruppo di test
for test_name in ["wilcoxon", "pearson", "spearman"]:
    p_col = f"p_{test_name}"
    test_df[f"p_{test_name}_BH"] = multipletests(
        test_df[p_col].to_numpy(),
        method="fdr_bh"
    )[1]
test_df["p_ttest_BH"] = multipletests(
    test_df["p_ttest"].to_numpy(),
    method="fdr_bh"
)[1]
test_df

In [ ]:
from utils.data import generate_synth_dataset

X, Y, Y_real = generate_synth_dataset(30000, {"l_r":False, "p_u": 0.3} , dataset_name="representational_bias", seed=42)
X

# TODO: modificare il train test splitting per stratificare per GRUPPO e Y
# TODO: scommentare il dizionario in config